![image.png](https://i.imgur.com/a3uAqnb.png)

# 🎥 UCF101 Video Classification Using 3D ResNet - Homework Assignment

In this homework, you will fine tune a **3D ResNet model for video classification** to recognize human actions from short video clips. This project is designed to give you practical experience working with spatiotemporal data using deep learning.

## 📌 Project Overview
- **Task**: Classify human actions from video clips
- **Architecture**: 3D ResNet for video classification
- **Dataset**: UCF101 action recognition dataset
- **Goal**: Build an accurate classification model using PyTorch

## 📚 Learning Objectives
By completing this assignment, you will:
- Learn how to process and sample frames from video files
- Fine tune a 3D ResNet model using PyTorch
- Train and evaluate models on the UCF101 dataset
- Analyze model performance on real-world video data

## 📦 Dataset Details
- The **UCF101** dataset contains 13,320 training videos across 101 action categories
- Each video clip captures a single action performed by a human in various environments
- Videos are processed into fixed-length clips (e.g., 16 or 32 frames) for training

## 🏷️ Labels Representation
- Each video is labeled with a class index from `0` to `100`
- Example classes include:
  - `Basketball`
  - `Horse Riding`
  - `Ice Dancing`
  - `Playing Guitar`
  - `Volleyball Spiking`

## 🔗 Reference
Original dataset link: [UCF101 Kaggle Dataset](https://www.kaggle.com/datasets/pevogam/ucf101)


## 1️⃣ Initial Setup and Library Installation

**Task**: Set up the environment and install necessary libraries.

In [ ]:
from IPython.display import clear_output

In [ ]:
# Incase you run this notebook outside colab (where the libraries aren't already pre-installed)

# %pip install torch
# %pip install matplotlib
# %pip install scikit-learn
# %pip install pandas
# %pip install numpy
# %pip install opencv-python

clear_output()

## 2️⃣ Import Libraries and Configuration

**Task**: Import all necessary libraries and set up configuration parameters.

**Requirements**:
- Import PyTorch and 3D ResNet model
- Import data processing libraries (pandas, sklearn, cv2)
- Import visualization libraries
- Set random seeds for reproducibility
- Configure hyperparameters with reasonable values

In [ ]:
import os
import cv2
import random
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, f1_score
import torch.nn.functional as F
import sklearn as sk


import torch
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
from torchvision.models.video import r3d_18
from tqdm.auto import tqdm

import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML

import kagglehub
from IPython.display import clear_output

In [ ]:
# TODO: Set random seeds for reproducibility

# TODO: Set device (GPU if available, otherwise CPU)

# TODO: Configure hyperparameters
BATCH_SIZE = None           # Batch size for training
LEARNING_RATE = None        # Learning rate for optimizer
NUM_EPOCHS = None           # Number of training epochs
FRAMES_PER_CLIP = None      # Number of frames to sample per video
NUM_CLASSES = None          # Number of action classes in UCF101
VALIDATION_SPLIT = None     # Validation split ratio
TEST_SPLIT = None           # Test split ratio

Using device: cuda


## 3️⃣ Data Loading and Exploration

**Task**: Download the UCF101 dataset and explore its structure.

**Requirements**:
- Download and load the dataset
- Display basic information about the data
- Load class mappings and video file paths
- Understand the dataset structure and labels

In [ ]:
path = kagglehub.dataset_download("pevogam/ucf101")

print("Path to dataset files:", path)

100%|██████████| 6.49G/6.49G [02:45<00:00, 42.1MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/pevogam/ucf101/versions/1


In [ ]:
# Load class name to index
classInd_txt = path + "/UCF101TrainTestSplits-RecognitionTask/ucfTrainTestlist/classInd.txt"
cls_df = pd.read_csv(classInd_txt, sep=" ", header=None,
                        names=["index", "class_name"])
cls_df.head()

,index,class_name
0,1,ApplyEyeMakeup
1,2,ApplyLipstick
2,3,Archery
3,4,BabyCrawling
4,5,BalanceBeam


In [ ]:
# TODO: Create class_to_idx and idx_to_class dictionaries
class_to_idx = None
idx_to_class = None

# TODO: Print dataset information

There are 101 classes.

{'ApplyEyeMakeup': 0, 'ApplyLipstick': 1, 'Archery': 2, 'BabyCrawling': 3, 'BalanceBeam': 4, 'BandMarching': 5, 'BaseballPitch': 6, 'Basketball': 7, 'BasketballDunk': 8, 'BenchPress': 9, 'Biking': 10, 'Billiards': 11, 'BlowDryHair': 12, 'BlowingCandles': 13, 'BodyWeightSquats': 14, 'Bowling': 15, 'BoxingPunchingBag': 16, 'BoxingSpeedBag': 17, 'BreastStroke': 18, 'BrushingTeeth': 19, 'CleanAndJerk': 20, 'CliffDiving': 21, 'CricketBowling': 22, 'CricketShot': 23, 'CuttingInKitchen': 24, 'Diving': 25, 'Drumming': 26, 'Fencing': 27, 'FieldHockeyPenalty': 28, 'FloorGymnastics': 29, 'FrisbeeCatch': 30, 'FrontCrawl': 31, 'GolfSwing': 32, 'Haircut': 33, 'Hammering': 34, 'HammerThrow': 35, 'HandstandPushups': 36, 'HandstandWalking': 37, 'HeadMassage': 38, 'HighJump': 39, 'HorseRace': 40, 'HorseRiding': 41, 'HulaHoop': 42, 'IceDancing': 43, 'JavelinThrow': 44, 'JugglingBalls': 45, 'JumpingJack': 46, 'JumpRope': 47, 'Kayaking': 48, 'Knitting': 49, 'LongJump': 50, 'Lunges':

In [ ]:
videos_root = path + "/UCF101/UCF-101"
paths_file = path + "/UCF101TrainTestSplits-RecognitionTask/ucfTrainTestlist/trainlist01.txt"


# TODO: Initialize lists for videos and labels
all_videos = []
all_labels = []

# TODO: Read training list file and extract video paths and labels

# TODO: Print total videos loaded

Total videos loaded: 9537


## 4️⃣ Data Preprocessing & Dataset Creation & data showing

**Task**: Create custom dataset class and prepare data splits.

**Requirements**:
- Implement video reading and frame sampling
- Create custom PyTorch Dataset class
- Split data into train/validation/test sets
- Apply appropriate transformations
- Create data loaders for batch processing
- Visualize clips of a video
- Show a video

In [ ]:
def read_video_opencv(video_path, frames_per_clip=16):
    # TODO: Open video using cv2.VideoCapture

    # TODO: Get total number of frames

    # TODO: Calculate frame indices to sample evenly

    # TODO: Read frames and convert BGR to RGB

    # TODO: Close video capture

    # TODO: Pad frames if too short

    # TODO: Convert to tensor and rearrange dimensions to (C, T, H, W)

    pass

In [ ]:
class UCF101Dataset(Dataset):
    def __init__(self, videos, labels, transform=None, frames_per_clip=16):
        # TODO: Initialize dataset with videos, labels, transforms, and frames_per_clip
        pass

    def __len__(self):
        # TODO: Return dataset length
        pass

    def __getitem__(self, idx):
        # TODO: Get video path and label for given index

        # TODO: Read video frames using read_video_opencv

        # TODO: Handle frame sampling and padding

        # TODO: Apply transforms if provided

        # TODO: Return clip and label
        pass

In [ ]:
# TODO: Split data into train/validation/test sets using train_test_split
# Use stratification to maintain class balance
train_videos, temp_videos, train_labels, temp_labels = None

val_videos, test_videos, val_labels, test_labels = None

# TODO: Print dataset sizes

Train size: 6675 6675
Train size: 1431 1431
Test size: 1431 1431


In [ ]:
# TODO: Define image transformations (resize and convert to float)
transform = None

# TODO: Create dataset instances for train, validation, and test
train_dataset = None
val_dataset = None
test_dataset = None

# TODO: Create data loaders with appropriate batch size and shuffling
train_loader = None
val_loader = None
test_loader = None

In [ ]:
# TODO: Test data loader by getting one batch
# Print batch shape and labels

# TODO: Get one sample from training dataset
# Print sample video shape and label with class name

Batch shape: torch.Size([8, 3, 16, 128, 128])
Labels: tensor([39, 46,  0, 13, 58, 14, 20, 62])
Sample video shape: torch.Size([3, 16, 128, 128])
Sample label: 71 (PushUps)


In [ ]:
def show_video_clip(tensor_clip, label='', fps=None):
    """
    Display a (C, T, H, W) tensor as an animation inline with a label.

    Parameters:
      tensor_clip (Tensor): A tensor of shape (C, T, H, W)
      label (str): Text label to display (e.g., class name)
      fps (int): Frames per second. If None, defaults to T (one second playback)
    """
    # Permute from (C, T, H, W) to (T, H, W, C)
    clip = tensor_clip.permute(1, 2, 3, 0).cpu().numpy()

    if fps is None:
        fps = clip.shape[0]  # Use T as default fps

    fig, ax = plt.subplots()
    ax.set_title(label)
    im = ax.imshow(clip[0])

    def update(frame_idx):
        im.set_array(clip[frame_idx])
        return [im]

    ani = animation.FuncAnimation(fig, update, frames=clip.shape[0], interval=1000 // fps, blit=True)
    plt.close(fig)

    video_html = ani.to_html5_video()
    video_html = video_html.replace("controls>", "controls autoplay muted>")
    return HTML(video_html)



In [ ]:
# TODO: Randomly get one video and show all its clips


In [ ]:
# TODO: show the video using show_video_clip


## 5️⃣ Model Architecture and Fine-Tuning Setup

**Task**: Set up 3D ResNet model for fine-tuning and define training functions.

**Requirements**:
- Load pre-trained 3D ResNet model
- Modify final layer for UCF101 classes
- Define training and evaluation functions
- Implement metrics calculation

In [ ]:
def build_model(num_classes):
    # TODO: Load pre-trained r3d_18 model

    # TODO: Replace final fully connected layer for UCF101 classes

    # TODO: Return modified model
    pass

def compute_metrics(preds, labels, num_classes):
    # TODO: Move tensors to CPU and get predicted labels

    # TODO: Calculate accuracy, precision, recall, and F1-score
    # Use sklearn metrics with macro averaging

    # TODO: Return metrics as percentages
    pass

In [ ]:
# TODO: Build model and move to device
model = None

# TODO: Initialize optimizer (Adam) with learning rate
optimizer = None

# TODO: Define loss function (CrossEntropyLoss)
criterion = None

# TODO: Print model information

/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=R3D_18_Weights.KINETICS400_V1`. You can also use `weights=R3D_18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/r3d_18-b3b3357e.pth" to /root/.cache/torch/hub/checkpoints/r3d_18-b3b3357e.pth
100%|██████████| 127M/127M [00:00<00:00, 159MB/s]


Model loaded on cuda
Total parameters: 33,218,085


## 6️⃣ Training and Validation Functions

**Task**: Implement comprehensive training and validation loops.

**Requirements**:
- Define training loop with gradient computation
- Implement validation loop with metrics tracking
- Monitor training progress with appropriate metrics

In [ ]:
def train_one_epoch(model, train_loader, optimizer, criterion, device, num_classes):
    # TODO: Set model to training mode

    # TODO: Initialize tracking variables

    # TODO: Create progress bar for training loop

    # TODO: Iterate through training batches
    for videos, labels in train_loader:
        # TODO: Move data to device

        # TODO: Zero gradients

        # TODO: Forward pass

        # TODO: Calculate loss

        # TODO: Backward pass and optimization step

        # TODO: Accumulate loss and predictions
        pass

    # TODO: Calculate metrics and return results
    pass

In [ ]:
@torch.no_grad()
def evaluate(model, val_loader, criterion, device, num_classes):
    # TODO: Set model to evaluation mode

    # TODO: Initialize tracking variables

    # TODO: Create progress bar for validation loop

    # TODO: Iterate through validation batches without gradient computation
    for videos, labels in val_loader:
        # TODO: Move data to device

        # TODO: Forward pass only

        # TODO: Calculate loss

        # TODO: Accumulate loss and predictions
        pass

    # TODO: Calculate metrics and return results
    pass

## 7️⃣ Training Loop with Validation

**Task**: Execute the complete training process with validation monitoring.

**Requirements**:
- Train model for specified epochs
- Track training and validation metrics
- Save best model based on validation performance
- Display training progress

In [ ]:
# TODO: Initialize tracking lists and best validation accuracy
train_losses = []
val_losses = []
best_val_acc = 0

# TODO: Training loop for specified epochs
for epoch in range(NUM_EPOCHS):
    # TODO: Print epoch information

    # TODO: Train for one epoch and get metrics

    # TODO: Append training loss and print results

    # TODO: Evaluate on validation set

    # TODO: Append validation loss and print results

    # TODO: Save best model if validation accuracy improved
    pass

## 8️⃣ Model Evaluation and Testing

**Task**: Evaluate the trained model on the test set and calculate classification metrics.

**Requirements**:
- Load best model for testing
- Make predictions on test set
- Calculate comprehensive classification metrics
- Analyze model performance

In [ ]:
# TODO: Load best model state dict

# TODO: Evaluate on test set

# TODO: Print comprehensive test results

## 9️⃣ Visualization and Analysis

**Task**: Create visualizations to analyze model performance and training progress.

**Requirements**:
- Plot training and validation loss curves
- Display sample predictions with confidence scores
- Analyze model performance across different classes

In [ ]:
# TODO: Create subplots for loss visualization

# TODO: Plot training and validation losses

# TODO: Add labels, legends, and formatting

# TODO: Display plots

In [ ]:
# TODO: Set model to evaluation mode

# TODO: Initialize list for sample predictions

# TODO: Make predictions on sample test videos
with torch.no_grad():
    for i in range(3):
        # TODO: Get sample video and true label

        # TODO: Prepare video batch for model input

        # TODO: Get model output and calculate probabilities

        # TODO: Get predicted label and confidence

        # TODO: Store prediction results
        pass

# TODO: Display prediction results with true vs predicted labels

## 📝 Evaluation Criteria

Your homework will be evaluated based on:

1. **Implementation Correctness (50%)**
   - Proper 3D ResNet model setup and fine-tuning
   - Correct video data preprocessing and dataset implementation
   - Working training loop with validation
   - Appropriate loss function and optimizer usage

2. **Model Performance (25%)**
   - Reasonable classification metrics (Accuracy, Precision, Recall, F1)
   - Convergence during training
   - Generalization to test set

3. **Code Quality (25%)**
   - Clean, readable code with proper structure
   - Efficient video processing implementation
   - Good coding practices and documentation

# Created by: Hassan Alsayhah